In [ ]:
import requests
import json
import time
import pandas as pd
from datetime import datetime, timezone, timedelta
from bs4 import BeautifulSoup
import re


def load_session_with_cookies(uid, cookies_path=r"C:\tongji\0 code\01_data_collection\weibo_cookies.json"):
    """
    加载Cookies并根据提供的真实Request Headers构建一个完整的、高仿真的请求头。
    uid: 用户的ID，用于动态生成正确的Referer。
    """
    try:
        with open(cookies_path, "r") as f:
            cookies_list = json.load(f)

        session = requests.Session()
        for cookie in cookies_list:
            session.cookies.set(cookie["name"], cookie["value"])

        xsrf_token = ""
        for cookie in cookies_list:
            if cookie["name"] == "XSRF-TOKEN":
                xsrf_token = cookie["value"]
                break

        headers = {
            "accept": "application/json, text/plain, */*",
            "accept-language": "zh-CN,zh;q=0.9",
            "client-version": "v2.47.120",
            "priority": "u=1, i",
            "referer": f"https://weibo.com/u/{uid}",
            "sec-ch-ua": '"Google Chrome";v="141", "Not?A_Brand";v="8", "Chromium";v="141"',
            "sec-ch-ua-mobile": "?0",
            "sec-ch-ua-platform": '"Windows"',
            "sec-fetch-dest": "empty",
            "sec-fetch-mode": "cors",
            "sec-fetch-site": "same-origin",
            "server-version": "v2025.09.29.1",
            "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36",
            "x-requested-with": "XMLHttpRequest",
            "x-xsrf-token": xsrf_token,
        }

        session.headers.update(headers)

        if not xsrf_token:
            print("警告: 未在Cookies中找到 XSRF-TOKEN，这可能导致请求失败。")

        print("Session、Cookies和高仿真Headers加载成功。")
        return session

    except FileNotFoundError:
        print(f"错误: {cookies_path} 文件未找到。请先运行登录代码获取Cookies。")
        return None
    except Exception as e:
        print(f"加载Session时发生未知错误: {e}")
        return None


In [ ]:

# --- 日期标准化辅助函数  ---
def parse_weibo_time(time_str, reference_date):
    # ... 此函数代码与之前版本完全相同，此处省略 ...
    now = datetime.now()
    if '分钟前' in time_str:
        minutes = int(re.search(r'(\d+)分钟前', time_str).group(1))
        post_time = now - timedelta(minutes=minutes)
        return post_time.strftime('%Y-%m-%d %H:%M:%S')
    if '小时前' in time_str:
        hours = int(re.search(r'(\d+)小时前', time_str).group(1))
        post_time = now - timedelta(hours=hours)
        return post_time.strftime('%Y-%m-%d %H:%M:%S')
    if '今天' in time_str:
        time_part = time_str.replace('今天', '').strip()
        date_part = now.strftime('%Y-%m-%d')
        return f"{date_part} {time_part}:00"
    if '月' in time_str and '日' in time_str:
        time_str = time_str.replace('月', '-').replace('日', '')
        year = reference_date.year
        return f"{year}-{time_str}:00"
    try:
        dt = datetime.strptime(time_str, '%Y-%m-%d %H:%M')
        return dt.strftime('%Y-%m-%d %H:%M:%S')
    except ValueError:
        try:
            dt = datetime.strptime(time_str, '%Y-%m-%d')
            return dt.strftime('%Y-%m-%d 00:00:00')
        except ValueError:
            return time_str


In [ ]:
# --- “工人”函数 (已完全修改为API JSON解析) ---
def scrape_weibo_by_hour(session, keyword, start_hour_str, end_hour_str, reference_date):
    """
    在指定的单一个小时范围内，爬取原创微博。
    修正1: 优化了点赞、评论、转发数的提取逻辑。
    修正2: 增加了页面内容重复性检测，防止因分页循环导致的死循环。
    """
    page = 1
    hourly_posts_data = []
    # previous_page_mids = set() # 用于存储上一页的微博ID，以检测重复

    # while page<2:
    while True:

        # 微博高级搜索的URL
        search_url = f"https://s.weibo.com/weibo?q={keyword}&typeall=1&suball=1&timescope=custom%3A{start_hour_str}%3A{end_hour_str}&Refer=g&page={page}"
        # print(search_url)

        
        try:
            # 微博搜索页最多只显示50页
            if page > 50:
                print(f"--- {start_hour_str} 已达到50页限制，停止爬取该小时 ---")
                break
            response = session.get(search_url, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')


            # --- 检测页面是否提示“未找到相关结果” ---
            if "抱歉，未找到相关结果。" in response.text:
                # print(response,response.text)

                print(f"--- {start_hour_str} 页面提示“未找到相关结果”，停止爬取该小时 ---")
                break

            posts = soup.find_all('div', {'class': 'card-wrap', 'action-type': 'feed_list_item'})

            # print(posts)

            # --- 检测页面循环：page=51跳回第一页 ---
            if not posts:
                print(f"--- {start_hour_str} 的第 {page} 页无数据，停止爬取该小时 ---")
                break

            current_page_mids = {post.get('mid') for post in posts if post.get('mid')}

            if page ==1:
                first_page_mids = current_page_mids
            elif current_page_mids == first_page_mids:
                print(f"--- {start_hour_str} 的第 {page} 页内容与首页重复，判定为末页，停止爬取该小时 ---")
                break
            # previous_page_mids = current_page_mids



            print(f"--- 正在爬取 {start_hour_str} 的第 {page} 页 ---")

            for post in posts:
                # print(post)
                post_id = post.get('mid')
                if not post_id:
                    continue

                # 区分原创微博和转发微博
                retweet_div = post.find('div', {'class': 'card-comment'})

                if retweet_div:
                    # 这是转发微博
                    post_type = 'retweet'
                    # 转发时自己添加的评论，是外层的 <p class='txt'>
                    # content_div = post.find('p', {'class': 'txt'})
                    # 1. 提取转发者的评论作为 'content'
                    main_content_div = post.find('div', class_='content')
                    if main_content_div:
                        content_p = main_content_div.find('p', attrs={'node-type': 'feed_list_content'}, recursive=False)
                        content = content_p.text.strip() if content_p else ''
                    
                    # 注意：如果转发时没写评论，content_div可能不存在或为空
                    # content = content_div.text.strip() if content_div and content_div.find('a') is None else ''
                    # 原始微博ID在 card-comment div 的 mid 属性中
                    # source_post_id = retweet_div.get('mid', '')

                    # 2. 从“展开”链接中提取 source_post_id
                    media_div = retweet_div.find(lambda tag: tag.has_attr('action-data'))
                    if media_div and media_div['action-data']:
                        mid_match = re.search(r"mid=(\d+)", media_div['action-data'])
                        if mid_match:
                            source_post_id = mid_match.group(1)
                    # 如果没有媒体模块，再尝试从“展开”链接提取
                    if not source_post_id:
                        unfold_link = retweet_div.find('a', {'action-type': 'fl_unfold'})
                        if unfold_link and unfold_link.get('href'):
                            href = unfold_link.get('href')
                            try:
                                source_post_id = href.split('/')[-1].split('?')[0]
                            except IndexError:
                                pass
                    
                    # 3. 提取原始微博的作者昵称
                    source_author_tag = retweet_div.find('a', class_='name')
                    if source_author_tag and source_author_tag.get('nick-name'):
                        source_author_name = source_author_tag.get('nick-name')
                    
                    # 在转发区块内部寻找图片和视频
                    post_clone = BeautifulSoup(str(post), 'html.parser')
                    retweet_block_in_clone = post_clone.find('div', class_='card-comment')
                    if retweet_block_in_clone:
                        retweet_block_in_clone.decompose() # 从副本中删除
                    # 在只剩下转发者内容的HTML中寻找媒体
                    search_scope = post_clone
                    
                else:
                    # 这是原创微博
                    post_type = 'original'
                    # 原创内容是外层的 <p class='txt'>
                    content_div = post.find('p', {'class': 'txt'})
                    content = content_div.text.strip() if content_div else ''
                    # 原创微博没有 source_post_id
                    source_post_id = ''
                    source_author_name = ''

                    # 原创微博在自身区块寻找图片和视频
                    search_scope = post



                user_info_div = post.find('a', {'class': 'name'})
                author_name = user_info_div.text.strip() if user_info_div else ''
                author_uid_link = user_info_div['href'] if user_info_div else ''
                author_uid = author_uid_link.split('/')[-1].split('?')[0] if author_uid_link else ''

                from_div = post.find('div', {'class': 'from'})
                post_time_str = from_div.find_all('a')[0].text.strip() if from_div and from_div.find_all('a') else ''
                standard_time = parse_weibo_time(post_time_str, reference_date)

                post_url = "https:" + from_div.find_all('a')[0]['href'] if from_div and from_div.find_all('a') else ''

                # content_div = post.find('p', {'class': 'txt'})
                # content = ''.join(content_div.find_all(string=True, recursive=False)).strip() if content_div else ''

                # --- 转发、评论、点赞数提取 ---
                shares, comments, likes = "0", "0", "0"
                stats_div = post.find('div', {'class': 'card-act'})
                if stats_div:
                    stats_list = stats_div.find('ul').find_all('li')
                    if len(stats_list) >= 3:
                        # 转发
                        share_text = stats_list[0].text.strip()
                        share_match = re.search(r'\d+', share_text)
                        shares = share_match.group(0) if share_match else '0'
                        # 评论
                        comment_text = stats_list[1].text.strip()
                        comment_match = re.search(r'\d+', comment_text)
                        comments = comment_match.group(0) if comment_match else '0'
                        # 点赞
                        like_text = stats_list[2].text.strip()
                        like_match = re.search(r'\d+', like_text)
                        likes = like_match.group(0) if like_match else '0'

                # --- 处理图片和视频 ---
                img_links = []
                video_cover_url = ''
                video_page_url = ''

                # 1. 检查是否存在图片模块
                # img_divs = post.find_all('div', {'class': 'media-piclist'})
                img_divs = search_scope.find_all('div', {'class': 'media-piclist'})
                if img_divs:
                    for div in img_divs:
                        imgs = div.find_all('img')
                        for img in imgs:
                            img_url = "https://image.baidu.com/search/down?url="+img.get('src', '').replace('wap180', 'large').replace('thumbnail', 'large')
                            img_links.append(img_url)
                
                # 2. 检查是否存在视频模块
                # video_div = post.find('div', {'class': 'media-box'})
                # if video_div:
                #     # 查找包含视频链接的<a>标签
                #     link_tag = video_div.find('a', attrs={'action-type': 'feed_list_video_click'})
                #     if link_tag:
                #         # 提取视频播放页链接
                #         video_page_url = link_tag.get('href', '')
                #         # 在<a>标签内查找封面图<img>标签
                #         img_tag = link_tag.find('img')
                #         if img_tag:
                #             video_cover_url =img_tag.get('src', '')

                # video_player_tag = post.find('video-player')
                video_player_tag = search_scope.find('video-player')

                if video_player_tag:
                    options_str = video_player_tag.get(':options', '')
                    if options_str:
                        # 提取视频封面和地址
                        poster_match = re.search(r"poster:\s*'([^']*)'", options_str)
                        address_match = re.search(r"address:\s*'([^']*)'", options_str)
                        
                        if poster_match:
                            video_cover_url = "https://image.baidu.com/search/down?url="+poster_match.group(1)
                        if address_match:
                            video_page_url = address_match.group(1)
                            # Ensure the URL has a protocol
                            if video_page_url.startswith('//'):
                                video_page_url = 'https:' + video_page_url
                else:
                    # 备用方法：有时视频信息在 media-box div 中
                    video_div = post.find('div', {'class': 'media-box'})
                    if video_div:
                        link_tag = video_div.find('a', attrs={'action-type': 'feed_list_video_click'})
                        if link_tag:
                            video_page_url = link_tag.get('href', '')
                            if video_page_url.startswith('//'):
                                video_page_url = 'https:' + video_page_url
                            
                            img_tag = link_tag.find('img')
                            if img_tag:
                                video_cover_url = img_tag.get('src', '')
                                if video_cover_url.startswith('//'):
                                    video_cover_url = "https://image.baidu.com/search/down?url="+'https:' + video_cover_url

                hourly_posts_data.append(
                    {
                        "post_id": post_id,
                        "content": content,
                        "created_at": standard_time,
                        "reposts_count": shares,
                        "comments_count": comments,
                        "likes_count": likes,  
                        "post_url": post_url,  # 微博链接
                        'image_urls': ', '.join(img_links),        # 图片链接
                        'video_cover_url': video_cover_url,     # 视频封面
                        'video_page_url': video_page_url,      # 视频页链接
                        "author_id": author_uid,  # 作者UID
                        "author_name": author_name, # 作者昵称
                        "post_type": post_type,   #是否为原创
                        "source_post_id": None if post_type == 'original' else source_post_id, # 原始微博ID
                        "source_author_name": None if post_type == 'original' else source_author_name # 原始微博作者昵称
                    }
                )

            page += 1
            time.sleep(5) # 尊重服务器，设置延迟

        except requests.exceptions.RequestException as e:
            print(f"爬取小时 {start_hour_str} 的第 {page} 页时发生网络错误: {e}")
            time.sleep(60) # 发生错误时，等待更长时间
            continue # 可以选择重试当前页
        except Exception as e:
            print(f"爬取小时 {start_hour_str} 的第 {page} 页时发生未知错误: {e}")
            break

    return hourly_posts_data

# --- 主调用函数 ---
# (此函数逻辑基本不变, 仅调用关系)
def run_hourly_search(uid, keyword, start_date_str, end_date_str,start_hour=0,end_hour=24):
    """
    主调用函数：接收一个日期范围，自动按天、再按小时切分任务，以突破50页限制。
    """
    all_data = []
    
    # 使用您的UID加载会话
    session = load_session_with_cookies(uid=uid) # 替换为你的微博UID
    if not session:
        print("Session加载失败，程序终止。")
        return

    try:
        start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
        end_date = datetime.strptime(end_date_str, '%Y-%m-%d')
    except ValueError:
        print("日期格式错误，请输入 'YYYY-MM-DD' 格式。")
        return
    
    current_day = start_date
    while current_day <= end_date:
        day_str = current_day.strftime("%Y-%m-%d")
        print(f"\n==================== 开始处理日期: {day_str} ====================")
        
        start_hour=start_hour
        end_hour=end_hour
        for hour in range(start_hour, end_hour):
            start_hour_str = f"{day_str}-{hour}"
            # 结束时间是下一小时的开始
            # next_hour_time = current_day + timedelta(hours=hour + 1)
            end_hour_str = f"{day_str}-{hour + 1}"
            
            # 调用更新后的 "工人" 函数, 传入当天的日期对象作为时间转换的参考

            hourly_data = scrape_weibo_by_hour(session, keyword, start_hour_str, end_hour_str, current_day)
            all_data.extend(hourly_data)
        
        print(f"日期 {day_str} 处理完毕。")
        current_day += timedelta(days=1)

    if not all_data:
        print("\n在指定日期范围内未找到任何符合条件的原创微博。")
        return

    df = pd.DataFrame(all_data)
    # 基于 post_id 去重
    df = df.drop_duplicates(subset=['post_id'])
    
    filename = f"weibo_search_{keyword}_{start_date_str}-{start_hour}_to_{end_date_str}-{end_hour}_final.csv"
    output_path = "C:\\tongji\\0 code\\00_data\\raw_weibo\\"  # 可以根据需要修改输出路径
    df.to_csv(output_path + filename, index=False, encoding='utf-8-sig')
    print(f"\n全部任务完成！共获取 {len(df[df['post_type'] == 'original'])} 条不重复的原创微博，{len(df[df['post_type'] == 'retweet'])} 条不重复的转发微博，数据已保存到 {output_path + filename}")

In [ ]:
for i in range(1,30):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2025-11-{i:02d}",
            end_date_str=f"2025-11-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2025-11-{i:02d} 时出错: {e}")  

In [ ]:
for i in range(1,31):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2025-10-{i:02d}",
            end_date_str=f"2025-10-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2025-10-{i:02d} 时出错: {e}")  

In [ ]:
for i in range(1,31):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2024-12-{i:02d}",
            end_date_str=f"2024-12-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2024-12-{i:02d} 时出错: {e}")  

In [ ]:
for i in range(1,32):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2025-01-{i:02d}",
            end_date_str=f"2025-01-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2025-01-{i:02d} 时出错: {e}")  

In [ ]:
for i in range(1,29):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2025-02-{i:02d}",
            end_date_str=f"2025-02-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2025-02-{i:02d} 时出错: {e}")  

In [ ]:
for i in range(1,32):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2025-03-{i:02d}",
            end_date_str=f"2025-03-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2025-03-{i:02d} 时出错: {e}")  

In [ ]:
for i in range(1,31):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2025-04-{i:02d}",
            end_date_str=f"2025-04-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2025-04-{i:02d} 时出错: {e}")  

In [ ]:
run_hourly_search(
    uid = '7801655101', 
    keyword="无限暖暖",
    start_date_str="2025-09-30",
    end_date_str="2025-09-30",
    start_hour=0,#包含
    end_hour=1 #不包含
)

In [ ]:
for i in range(1,31):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2025-06-{i:02d}",
            end_date_str=f"2025-06-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2025-06-{i:02d} 时出错: {e}")  

In [ ]:
for i in range(1,32):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2025-08-{i:02d}",
            end_date_str=f"2025-08-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2025-08-{i:02d} 时出错: {e}")  

for i in range(1,32):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2025-07-{i:02d}",
            end_date_str=f"2025-07-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2025-07-{i:02d} 时出错: {e}")  

In [ ]:
for i in range(1,31):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2025-06-{i:02d}",
            end_date_str=f"2025-06-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2025-06-{i:02d} 时出错: {e}")  

test part

In [ ]:
for i in range(1,32):
    try:
        run_hourly_search(
            uid = '7801655101', 
            keyword="无限暖暖",
            start_date_str=f"2025-05-{i:02d}",
            end_date_str=f"2025-05-{i:02d}"
        )
    except Exception as e:
        print(f"处理日期 2025-05-{i:02d} 时出错: {e}")  

In [ ]:
session = load_session_with_cookies(uid='7801655101') # 替换为你的微博UID
uid = '7801655101', 
keyword="无限暖暖",
start_hour_str = "2025-09-30-0"
end_hour_str = "2025-09-30-1"
page = 1
search_url = f"https://s.weibo.com/weibo?q={keyword}&typeall=1&suball=1&timescope=custom%3A{start_hour_str}%3A{end_hour_str}&Refer=g&page={page}"
response = session.get(search_url, timeout=10)
print(response.text)
soup = BeautifulSoup(response.text, 'html.parser')
posts = soup.find_all('div', {'class': 'card-wrap', 'action-type': 'feed_list_item'})
print(posts)


In [ ]:
session = load_session_with_cookies(uid='7801655101') # 替换为你的微博UID
uid = '7801655101', 
keyword="无限暖暖",
start_hour_str = "2025-09-30-0"
end_hour_str = "2025-09-30-1"
page = 5
search_url = f"https://s.weibo.com/weibo?q={keyword}&typeall=1&suball=1&timescope=custom%3A{start_hour_str}%3A{end_hour_str}&Refer=g&page={page}"
response = session.get(search_url, timeout=10)
print(response.text)
soup = BeautifulSoup(response.text, 'html.parser')
posts = soup.find_all('div', {'class': 'card-wrap', 'action-type': 'feed_list_item'})
print(posts)
